# Notebook 2 — Fine-tuning de modelos YOLO para detección de EPP

**TP Visión por Computadora II — CEIA FIUBA**

En este notebook se entrenan tres variantes de YOLO mediante fine-tuning sobre el dataset
Construction-PPE descargado en el notebook anterior.

Objetivos:
1. Entrenar **YOLOv8n** (nano) — modelo liviano, orientado a velocidad
2. Entrenar **YOLOv8m** (medium) — mayor capacidad, más lento
3. Entrenar **YOLOv11n** (nano, arquitectura v11) — nueva generación con mejoras en el cuello de botella
4. Evaluar los tres modelos en el set de validación y medir latencia de inferencia
5. Exportar métricas para la comparación del notebook 3

Los tres modelos parten de pesos preentrenados en COCO (transfer learning).
Como se vio en clase, el fine-tuning permite aprovechar las features genéricas aprendidas
en un dataset grande y adaptarlas a un dominio específico con relativamente pocas épocas de entrenamiento.

In [ ]:
# Detectar si se está ejecutando en Google Colab
import importlib
EN_COLAB = importlib.util.find_spec("google.colab") is not None

if EN_COLAB:
    print("Entorno: Google Colab")

    # Montar Google Drive para persistir archivos entre sesiones
    from google.colab import drive
    drive.mount("/content/drive")

    # Moverse a la carpeta del proyecto en Drive
    import os
    os.chdir("/content/drive/MyDrive/vision_computadora_2")
    print(f"Directorio de trabajo: {os.getcwd()}")

    # Instalar dependencias
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "ultralytics", "roboflow"])
else:
    print("Entorno: local")

## 1. Configuración del entrenamiento

Se cargan los parámetros del dataset generados en el notebook 1 y se definen los hiperparámetros
de entrenamiento. Los valores se ajustan automáticamente según el dispositivo disponible:
en GPU se usan resoluciones y batches más grandes, mientras que en CPU se reducen para que
el entrenamiento sea viable en tiempos razonables.

In [ ]:
import json
import shutil
import time
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO

# Configuración del dataset (generada en el notebook 1)
with open("../data/dataset_config.json") as f:
    config_dataset = json.load(f)

ruta_yaml_dataset = config_dataset["dataset_yaml"]
nombres_clases = config_dataset["class_names"]
cantidad_clases = config_dataset["num_classes"]

# Hiperparámetros de entrenamiento — se adaptan al dispositivo disponible
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
tamanio_imagen = 640 if dispositivo == "cuda" else 416
tamanio_batch = 16 if dispositivo == "cuda" else 4
cantidad_epocas = 50 if dispositivo == "cuda" else 15
num_workers = 4 if dispositivo == "cuda" else 0

carpeta_runs = Path("../runs")
carpeta_modelos = Path("../models")
carpeta_modelos.mkdir(exist_ok=True)

print(f"Dispositivo:        {dispositivo}")
print(f"Tamaño de imagen:   {tamanio_imagen}x{tamanio_imagen}")
print(f"Tamaño de batch:    {tamanio_batch}")
print(f"Épocas:             {cantidad_epocas}")
print(f"Dataset YAML:       {ruta_yaml_dataset}")
print(f"Clases ({cantidad_clases}): {nombres_clases}")

## 2. Entrenamiento de YOLOv8n (Nano)

YOLOv8n es la variante más liviana de la familia YOLOv8 (~3M parámetros).
Al cargar `yolov8n.pt`, se descargan automáticamente los pesos preentrenados en COCO (80 clases).
Ultralytics reemplaza la última capa de detección para adaptarla a nuestras 10 clases
y congela la capa DFL (Distribution Focal Loss), manteniendo el resto de la red entrenable.

Se utiliza `patience=10` para early stopping: si la métrica de validación no mejora
durante 10 épocas consecutivas, el entrenamiento se detiene automáticamente.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv8n (nano)")
print("=" * 60)

modelo_v8n = YOLO("yolov8n.pt")

resultados_v8n = modelo_v8n.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=tamanio_batch,
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov8n_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

# Copiar el mejor modelo a la carpeta de modelos finales
ruta_mejor_v8n = carpeta_runs / "yolov8n_ppe" / "weights" / "best.pt"
shutil.copy(ruta_mejor_v8n, carpeta_modelos / "yolov8n_best.pt")
print(f"\nYOLOv8n guardado en: {carpeta_modelos / 'yolov8n_best.pt'}")

  Entrenando YOLOv8n (nano)
Ultralytics 8.4.33 🚀 Python-3.11.13 torch-2.11.0 CPU (Apple M3 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/mferrari/git/fiuba/vision_computadora_2/data/construction-ppe/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_ppe, nbs=64, nms=False, opset=None, optimize=Fa

## 3. Entrenamiento de YOLOv8m (Medium)

YOLOv8m tiene mayor capacidad (~25M parámetros) que la variante nano.
Al tener más capas y filtros, puede capturar patrones más complejos, pero a costa de
mayor tiempo de entrenamiento e inferencia.

Se reduce el tamaño de batch a la mitad para evitar problemas de memoria,
especialmente relevante en CPU donde la RAM es el factor limitante.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv8m (medium)")
print("=" * 60)

modelo_v8m = YOLO("yolov8m.pt")

resultados_v8m = modelo_v8m.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=max(2, tamanio_batch // 2),  # batch reducido por el mayor tamaño del modelo
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov8m_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

ruta_mejor_v8m = carpeta_runs / "yolov8m_ppe" / "weights" / "best.pt"
shutil.copy(ruta_mejor_v8m, carpeta_modelos / "yolov8m_best.pt")
print(f"\nYOLOv8m guardado en: {carpeta_modelos / 'yolov8m_best.pt'}")

## 4. Entrenamiento de YOLOv11n (Nano, arquitectura v11)

YOLOv11 (también referido como YOLO11 en Ultralytics) introduce mejoras arquitectónicas
respecto a v8, particularmente en el cuello de botella (neck) y el cabezal de detección.
Mantiene un tamaño similar a YOLOv8n pero con mejor eficiencia computacional.

Se entrena con los mismos hiperparámetros que YOLOv8n para que la comparación sea justa.

In [ ]:
print("=" * 60)
print("  Entrenando YOLOv11n (YOLO11 nano)")
print("=" * 60)

modelo_v11n = YOLO("yolo11n.pt")

resultados_v11n = modelo_v11n.train(
    data=ruta_yaml_dataset,
    epochs=cantidad_epocas,
    imgsz=tamanio_imagen,
    batch=tamanio_batch,
    device=dispositivo,
    workers=num_workers,
    project=str(carpeta_runs),
    name="yolov11n_ppe",
    exist_ok=True,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
)

ruta_mejor_v11n = carpeta_runs / "yolov11n_ppe" / "weights" / "best.pt"
shutil.copy(ruta_mejor_v11n, carpeta_modelos / "yolov11n_best.pt")
print(f"\nYOLOv11n guardado en: {carpeta_modelos / 'yolov11n_best.pt'}")

## 5. Evaluación en el set de validación

Se evalúan los tres modelos entrenados sobre el set de validación utilizando las métricas
estándar de detección de objetos:

- **mAP50**: Mean Average Precision con umbral de IoU = 0.50
- **mAP50-95**: Promedio de mAP en umbrales de IoU de 0.50 a 0.95 (métrica principal de COCO)
- **Precision**: Proporción de detecciones correctas sobre el total de detecciones
- **Recall**: Proporción de objetos reales que fueron detectados

Los resultados se exportan a un archivo JSON para su análisis en el notebook 3.

In [ ]:
rutas_modelos = {
    "YOLOv8n": carpeta_modelos / "yolov8n_best.pt",
    "YOLOv8m": carpeta_modelos / "yolov8m_best.pt",
    "YOLOv11n": carpeta_modelos / "yolov11n_best.pt",
}

resultados_evaluacion = {}

for nombre_modelo, ruta_modelo in rutas_modelos.items():
    if not ruta_modelo.exists():
        print(f"  {nombre_modelo}: no encontrado, se omite.")
        continue

    print(f"\nEvaluando {nombre_modelo}...")
    modelo = YOLO(str(ruta_modelo))

    inicio = time.time()
    metricas = modelo.val(
        data=ruta_yaml_dataset,
        imgsz=tamanio_imagen,
        device=dispositivo,
        workers=num_workers,
        verbose=False,
    )
    tiempo_evaluacion = time.time() - inicio

    resultados_evaluacion[nombre_modelo] = {
        "map50": round(float(metricas.box.map50), 4),
        "map50_95": round(float(metricas.box.map), 4),
        "precision": round(float(metricas.box.mp), 4),
        "recall": round(float(metricas.box.mr), 4),
        "val_time_s": round(tiempo_evaluacion, 1),
    }

    r = resultados_evaluacion[nombre_modelo]
    print(f"  mAP50:      {r['map50']:.4f}")
    print(f"  mAP50-95:   {r['map50_95']:.4f}")
    print(f"  Precision:  {r['precision']:.4f}")
    print(f"  Recall:     {r['recall']:.4f}")

# Guardar resultados para el notebook 3
with open("../data/eval_results.json", "w") as f:
    json.dump(resultados_evaluacion, f, indent=2)

print("\nResultados guardados en data/eval_results.json")

## 6. Benchmark de latencia de inferencia

Se mide el tiempo de inferencia de cada modelo para estimar su viabilidad en aplicaciones
de tiempo real. Se utiliza una imagen sintética del mismo tamaño que las imágenes de entrenamiento
y se promedian 20 ejecuciones (descartando 3 de warm-up) para obtener una medición estable.

La latencia se reporta en milisegundos y se calcula también los FPS (frames por segundo).

In [ ]:
# Imagen sintética para medir latencia (mismo tamaño que las de entrenamiento)
imagen_sintetica = np.random.randint(0, 255, (tamanio_imagen, tamanio_imagen, 3), dtype=np.uint8)
cantidad_mediciones = 20

resultados_latencia = {}

for nombre_modelo, ruta_modelo in rutas_modelos.items():
    if not ruta_modelo.exists():
        continue

    modelo = YOLO(str(ruta_modelo))

    # Warm-up: las primeras ejecuciones son más lentas por inicialización interna
    for _ in range(3):
        _ = modelo.predict(imagen_sintetica, verbose=False, device=dispositivo)

    # Medición real
    tiempos_ms = []
    for _ in range(cantidad_mediciones):
        t0 = time.perf_counter()
        _ = modelo.predict(imagen_sintetica, verbose=False, device=dispositivo)
        tiempos_ms.append((time.perf_counter() - t0) * 1000)

    resultados_latencia[nombre_modelo] = {
        "mean_ms": round(np.mean(tiempos_ms), 1),
        "std_ms": round(np.std(tiempos_ms), 1),
        "p50_ms": round(np.percentile(tiempos_ms, 50), 1),
        "p95_ms": round(np.percentile(tiempos_ms, 95), 1),
        "fps": round(1000 / np.mean(tiempos_ms), 1),
    }

    r = resultados_latencia[nombre_modelo]
    print(f"{nombre_modelo}: {r['mean_ms']:.1f} ms +/- {r['std_ms']:.1f} | {r['fps']:.1f} FPS")

# Guardar latencias para el notebook 3
with open("../data/latency_results.json", "w") as f:
    json.dump(resultados_latencia, f, indent=2)

print("\nLatencias guardadas en data/latency_results.json")

## Resumen

En este notebook se entrenaron tres modelos YOLO mediante fine-tuning sobre el dataset Construction-PPE:

| Modelo | Parámetros | Características |
|--------|-----------|-----------------|
| YOLOv8n | ~3M | Liviano, rápido, ideal para edge/tiempo real |
| YOLOv8m | ~25M | Mayor capacidad, mejor precisión potencial |
| YOLOv11n | ~3M | Nueva arquitectura, mejoras en neck y head |

Los pesos entrenados se guardaron en `models/` y las métricas de evaluación y latencia
en `data/eval_results.json` y `data/latency_results.json` respectivamente.

**Siguiente paso:** Notebook 3 — Comparación cuantitativa de los tres modelos.

## 7. Descarga de artefactos (solo Colab)

Si se ejecutó en Google Colab sin montar Google Drive, los archivos se pierden al cerrar la sesión.
Si se montó Drive, los archivos ya están persistidos. La siguiente celda permite descargarlos
al navegador de ser necesario.

In [ ]:
if EN_COLAB:
    from google.colab import files

    archivos_a_descargar = [
        carpeta_modelos / "yolov8n_best.pt",
        carpeta_modelos / "yolov8m_best.pt",
        carpeta_modelos / "yolov11n_best.pt",
        Path("../data/eval_results.json"),
        Path("../data/latency_results.json"),
    ]

    for archivo in archivos_a_descargar:
        if archivo.exists():
            print(f"Descargando {archivo.name}...")
            files.download(str(archivo))
        else:
            print(f"  {archivo.name} no encontrado, se omite.")
else:
    print("Entorno local — los archivos ya estan en disco.")